In [2]:
with open('input.txt','r', encoding='utf-8') as f:
    text = f.read()

print('length of dataset in characters', (len(text)))

length of dataset in characters 1115394


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = {char:i for i, char in enumerate(chars)}
itos = {i:char for char, i in stoi.items()}
# encode ---> take a string, output a list of integers:
encode = lambda s : [stoi[c] for c in s]
# decode ---> take a list of integers, iutput a string
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [5]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)

torch.Size([1115394]) <built-in method type of Tensor object at 0x11145f200>


In [6]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is: {context} | the target: {target}")

when input is: tensor([18]) | the target: 47
when input is: tensor([18, 47]) | the target: 56
when input is: tensor([18, 47, 56]) | the target: 57
when input is: tensor([18, 47, 56, 57]) | the target: 58
when input is: tensor([18, 47, 56, 57, 58]) | the target: 1
when input is: tensor([18, 47, 56, 57, 58,  1]) | the target: 15
when input is: tensor([18, 47, 56, 57, 58,  1, 15]) | the target: 47
when input is: tensor([18, 47, 56, 57, 58,  1, 15, 47]) | the target: 58


In [9]:
batch_size = 4 # how many independet sequences will we process in parallel?
block_size = 8 # what is the maximum context length for prediction?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 47, 58,  8,  0,  0, 18, 47],
        [52, 39, 41, 50, 43, 57,  6,  0],
        [39, 42,  1, 53, 52,  1, 58, 53],
        [ 1, 45, 53,  1, 61, 47, 58, 46]])
targets:
torch.Size([4, 8])
tensor([[47, 58,  8,  0,  0, 18, 47, 56],
        [39, 41, 50, 43, 57,  6,  0, 32],
        [42,  1, 53, 52,  1, 58, 53,  1],
        [45, 53,  1, 61, 47, 58, 46,  1]])
-----
when input is [1] the target: 47
when input is [1, 47] the target: 58
when input is [1, 47, 58] the target: 8
when input is [1, 47, 58, 8] the target: 0
when input is [1, 47, 58, 8, 0] the target: 0
when input is [1, 47, 58, 8, 0, 0] the target: 18
when input is [1, 47, 58, 8, 0, 0, 18] the target: 47
when input is [1, 47, 58, 8, 0, 0, 18, 47] the target: 56
when input is [52] the target: 39
when input is [52, 39] the target: 41
when input is [52, 39, 41] the target: 50
when input is [52, 39, 41, 50] the target: 43
when input is [52, 39, 41, 50, 43] the target: 57
when input is [52, 39, 41, 50,

In [10]:
print(xb) # our input to the transformer

tensor([[ 1, 47, 58,  8,  0,  0, 18, 47],
        [52, 39, 41, 50, 43, 57,  6,  0],
        [39, 42,  1, 53, 52,  1, 58, 53],
        [ 1, 45, 53,  1, 61, 47, 58, 46]])


In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of ints
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens): # this is very general for a bigram model, because we are feeding the who string we have generated so far and looking only at the last index
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions:
            logits, loss = self(idx) # the loss will be ignored, because we have no targets
            # focus only on the last time step:
            logits = logits[:, -1, :] # becomes (B, C)
            # apply the softmax to get probs:
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution:
            idx_nex = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append samples index to the running sequence:
            idx = torch.cat((idx, idx_nex), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long) # the first char will be the new line ('/n') character
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.5586, grad_fn=<NllLossBackward0>)

k GI&ps:EGb'PL&WEFz3XwvJw!a,Gbh?DZuM'&kkkZ!sa
g3WulLQbIIaQ:EDcGb'kB!.!,VHDoDYG,P
nO;kYF
WNjB
wTtsb'f


In [12]:
# create a Pytorch optimizer:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [13]:
batch_size = 32
for steps in range(10000):

    # sample a batch of data:
    xb, yb = get_batch('train')

    # evaluate the loss:
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.3584792613983154


In [14]:
print(decode(m.generate(idx, max_new_tokens=400)[0].tolist()))


IN tacchedimp andilouGBOWheithacak penghngury neay, tavesiawak kefo hourabecime mound
nth m, b'do frsheaneay an tasther Fouche
GLUERINGr tofo an Soor y ond.
I tit y$we Westhany,
Y t py pppthiereppla cond?
TETham LOMay s fe te p ked t my? esomburoullow mind I' is witirs d thist usas hiveen s:
thadio owhetst ta bjes t be,

Awfinn re larror, w,


OMORas hileg, ereagenaghowhodyofathe coThate fitel s n


In [21]:
B, T, C = 4, 8, 32 # batch / time / channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 32])

In [22]:
# version 1:
xbow = torch.zeros((B, T, C)) # bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

In [23]:
# version 2:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (T, T) @ (B, T, C) ---> Pytorch will create a batch dimension for wei (B, T, T) this means for each batch_element we will do (T, T) @ (T, C) ---> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [24]:
# version 3 (using softmax)
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # apply softmax for each row
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [29]:
# version 4 ---> self attention
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# let's see a single Head perform self-attention:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])